In [1]:
import glob
import torch
import torch.nn as nn
import numpy as np
from torch.utils.data import DataLoader
from dataset import PendulumDataset
from model import LSTMModel
import torch.optim as optim
import random

seed = 0
torch.manual_seed(seed)
np.random.seed(seed)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [2]:
all_traj = np.load("double_pendulum_trajs.npy")  # (N, T, 4)
trajectories = [all_traj[i] for i in range(all_traj.shape[0])]

random.shuffle(trajectories)

N = len(trajectories)
n_train = int(0.7 * N)
n_val   = int(0.15 * N)

train_trajs = trajectories[:n_train]
val_trajs   = trajectories[n_train:n_train + n_val]
test_trajs  = trajectories[n_train + n_val:]

seq_len = 100

train_ds = PendulumDataset(train_trajs, seq_len)
val_ds   = PendulumDataset(val_trajs, seq_len,
                           mean=train_ds.mean, std=train_ds.std)
test_ds  = PendulumDataset(test_trajs, seq_len,
                           mean=train_ds.mean, std=train_ds.std)

batch_size = 50
train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=batch_size, shuffle=False)
test_loader  = DataLoader(test_ds,  batch_size=batch_size, shuffle=False)


/home/s2273998/mlp_project/dataset.py:27: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:253.)
  self.x = torch.tensor(self.x, dtype=torch.float32)


In [3]:
input_size = 4
hidden_layer_size = 128
output_size = 4
num_layers = 2
learning_rate = 0.001
num_epoch = 200

model = LSTMModel(input_size, hidden_layer_size, output_size, num_layers).to(device)
loss_function = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

train_losses = []
val_losses = []

def train_model(model, train_loader, val_loader, num_epochs):
    for epoch in range(num_epochs):
        model.train()
        train_loss = 0
        for sequences, targets in train_loader:
            sequences = sequences.to(device)
            targets = targets.to(device)
            
            optimizer.zero_grad()
            pred = model(sequences) 
            loss = loss_function(pred, targets)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()

        train_loss /= len(train_loader)
        train_losses.append(train_loss)
        model.eval()
        val_loss = 0 
        with torch.no_grad():
            for sequences, targets in val_loader:
                sequences = sequences.to(device)
                targets = targets.to(device)

                pred = model(sequences)
                loss = loss_function(pred, targets)
                val_loss += loss.item()
        
        val_loss /= len(val_loader)
        val_losses.append(val_loss)
        if (epoch + 1) % 10 == 0:
            print(f"Epoch {epoch+1}/{num_epochs} | train loss: {train_loss:.6f} | val loss: {val_loss:.6f}")
        
        
train_model(model, train_loader, val_loader, num_epoch)

Epoch 10/300 | train loss: 0.014557 | val loss: 0.004947
Epoch 20/300 | train loss: 0.013998 | val loss: 0.004504
Epoch 30/300 | train loss: 0.013757 | val loss: 0.004147
Epoch 40/300 | train loss: 0.013615 | val loss: 0.003870
Epoch 50/300 | train loss: 0.013434 | val loss: 0.004170
Epoch 60/300 | train loss: 0.013366 | val loss: 0.003907
Epoch 70/300 | train loss: 0.013284 | val loss: 0.003563
Epoch 80/300 | train loss: 0.013101 | val loss: 0.003465
Epoch 90/300 | train loss: 0.013064 | val loss: 0.003784
Epoch 100/300 | train loss: 0.012968 | val loss: 0.003275
Epoch 110/300 | train loss: 0.012845 | val loss: 0.003811
Epoch 120/300 | train loss: 0.012721 | val loss: 0.003131
Epoch 130/300 | train loss: 0.012770 | val loss: 0.003384
Epoch 140/300 | train loss: 0.012657 | val loss: 0.003757
Epoch 150/300 | train loss: 0.012567 | val loss: 0.003243
Epoch 160/300 | train loss: 0.012458 | val loss: 0.003579
Epoch 170/300 | train loss: 0.012528 | val loss: 0.004089
Epoch 180/300 | train l

In [4]:
def evaluate_one_step(model, test_loader, tol=0.05):
    model.eval()
    total_loss = 0
    total_correct = 0
    total_samples = 0
    criterion = nn.MSELoss(reduction='sum')

    with torch.no_grad():
        for sequences, targets in test_loader:
            sequences = sequences.to(device)
            targets = targets.to(device)

            preds = model(sequences)
            loss = criterion(preds, targets)
            total_loss += loss.item()

            # Absolute error per state variable
            abs_error = torch.abs(preds - targets)

            # A prediction is "accurate" if ALL 4 variables are within tolerance
            correct = (abs_error < tol).all(dim=1)
            total_correct += correct.sum().item()
            total_samples += targets.size(0)

    mse = total_loss / (total_samples * targets.size(1))
    rmse = mse ** 0.5
    accuracy = total_correct / total_samples * 100

    print(f"Test MSE: {mse:.6f}")
    print(f"Test RMSE: {rmse:.6f}")
    print(f'Accuracy (|error| < {tol} for all states): {accuracy:.2f}%')
    
evaluate_one_step(model, test_loader)

Test MSE: 0.002296
Test RMSE: 0.047920
Accuracy (|error| < 0.05 for all states): 98.07%


In [ ]:
def teacher_forcing(model, train_loader, test_loader, num_epochs, pred_len):
    for epoch in range(num_epochs):
        model.train()
        train_loss = 0
    
        for sequences, targets in train_loader:
            sequences, targets = sequences.to(device), targets.to(device)
            batch_size, seq_len, input_dim = sequences.size()
            optimizer.zero_grad()
            
            outputs = torch.zeros(batch_size, pred_len, 4).to(device)
            input_t = sequences[:, -1, :] 
            
            for t in range(pred_len):
                output_t, _ = model(input_t.unsqueeze(1))
                outputs[:, t, :] = output_t
                input_t = targets[:, t, :]
                
            loss = loss_function(output, targets)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()

        train_loss /= len(train_loader)
        train_losses.append(train_loss)
        